# Practical Session 04
## Part I - Instructor-led mini-lab
### Example 1 - Hidden mutation in a preprocessing helper

In this example, the numerical problem is simple: we extract a window from a signal and centre it.  
The important point is not the formula, but the data flow. Does the helper create a new object, or does it modify an object that is still needed later?

In [ ]:
import json
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(42)

time = np.linspace(0.0, 8.0, 400)
raw_signal = np.exp(-0.25 * time) * np.sin(3.0 * time)
raw_signal += 0.04 * rng.normal(size=time.size)

raw_reference = raw_signal.copy()

# This is a view into raw_signal, not an independent signal.
window = raw_signal[80:180]
window -= window.mean()

changed = ~np.isclose(raw_signal, raw_reference)

print("raw_signal unchanged?", np.allclose(raw_signal, raw_reference))
print("number of changed samples:", changed.sum())
print("first changed indices:", np.flatnonzero(changed)[:6])

The array `window` was only a view into `raw_signal`. The centering operation modified that view in place, so the original signal changed as well.

A safer interface should make the choice explicit: either return a new centred array, or clearly name the function as an in-place operation.

In [ ]:
raw_signal = raw_reference.copy()

def centered(values):
    "Return a centred result without modifying the input array."
    return values - values.mean()


def center_in_place(values):
    "Centre the input array in place and return the same object."
    values -= values.mean()
    return values


safe_window = centered(raw_signal[80:180])

print("after centered: raw_signal unchanged?", np.allclose(raw_signal, raw_reference))

independent_window = raw_signal[80:180].copy()
center_in_place(independent_window)

print("after center_in_place on a copy: raw_signal unchanged?", np.allclose(raw_signal, raw_reference))
print("safe_window mean:", safe_window.mean())
print("independent_window mean:", independent_window.mean())

Questions:

1. Which line in the first version changed the original signal?
2. Why is the name `center_in_place` more honest than `center`?
3. When could an in-place update be a good choice in a scientific code?

### Example 2 - Explicit interfaces for a small numerical experiment

Now we build a very small experiment with visible state: a configuration object, an explicit random generator, pure helper functions, and diagnostics returned as values.

In [ ]:
@dataclass(frozen=True)
class SignalConfig:
    tmin: float = 0.0
    tmax: float = 10.0
    nt: int = 500
    damping: float = 0.22
    omega: float = 3.5
    noise_sigma: float = 0.06

    def __post_init__(self):
        if self.nt < 2:
            raise ValueError("nt must be at least 2")
        if self.tmax <= self.tmin:
            raise ValueError("tmax must be larger than tmin")
        if self.noise_sigma < 0.0:
            raise ValueError("noise_sigma must be non-negative")


def make_time_grid(config):
    return np.linspace(config.tmin, config.tmax, config.nt)


def clean_signal(t, config):
    return np.exp(-config.damping * t) * np.sin(config.omega * t)


def add_noise(signal, *, sigma, rng):
    noise = sigma * rng.normal(size=signal.shape)
    return signal + noise


def signal_diagnostics(t, signal):
    assert t.ndim == 1
    assert signal.shape == t.shape
    assert np.isfinite(signal).all()

    rms = np.sqrt(np.mean(signal**2))
    i_peak = np.argmax(np.abs(signal))

    return {
        "rms": rms,
        "t_peak_abs": t[i_peak],
        "max_abs": np.abs(signal[i_peak]),
        "integral_abs": np.trapezoid(np.abs(signal), t),
    }


config = SignalConfig()
rng = np.random.default_rng(123)

t = make_time_grid(config)
signal_clean = clean_signal(t, config)
signal_measured = add_noise(signal_clean, sigma=config.noise_sigma, rng=rng)

diagnostics = signal_diagnostics(t, signal_measured)
diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.2))

ax.plot(t, signal_measured, ".", markersize=3, label="measured")
ax.plot(t, signal_clean, linewidth=2, label="clean model")
ax.axvline(diagnostics["t_peak_abs"], linestyle="--", label="largest |signal|")

ax.set_xlabel(r"time $t$")
ax.set_ylabel(r"signal")
ax.set_title(f"RMS = {diagnostics['rms']:.3f}")
ax.grid(True, alpha=0.3)
ax.legend()

Questions:

1. Which scientific choices are stored in `SignalConfig`?
2. Why is `rng` passed from the outside instead of created inside `add_noise`?
3. Which functions are pure computations, and which object carries the state of the experiment?

### Example 3 - A labelled table is more than a two-dimensional array

In this example, every row is one event-like measurement. Some columns are measured quantities, while other columns are labels or metadata keys. pandas keeps these meanings close to the data.

In [ ]:
rng = np.random.default_rng(10)

run_ids = np.array(["run_001", "run_002", "run_003", "run_004"])
detectors = np.array(["A", "B", "C"])

runs_raw = pd.DataFrame({
    "run_id": run_ids,
    "field_T": [0.10, 0.20, 0.30, 0.40],
    "temperature_K": [299.8, 300.1, 300.3, 300.0],
})

n_events = 600

events_raw = pd.DataFrame({
    "event_id": np.arange(n_events),
    "run_id": rng.choice(run_ids, size=n_events),
    "detector": rng.choice(detectors, size=n_events),
    "energy_GeV": rng.gamma(shape=4.0, scale=1.2, size=n_events),
    "chi2": rng.gamma(shape=1.5, scale=0.8, size=n_events),
    "signal": rng.normal(loc=12.0, scale=2.0, size=n_events),
    "exposure_s": rng.uniform(0.5, 2.0, size=n_events),
})

# A few deliberately problematic rows.
events_raw.loc[5, "signal"] = np.nan
events_raw.loc[17, "energy_GeV"] = -2.0
events_raw.loc[33, "exposure_s"] = 0.0
events_raw.loc[44, "chi2"] = np.inf

events_raw["run_id"] = events_raw["run_id"].astype("string")
events_raw["detector"] = events_raw["detector"].astype("category")
runs_raw["run_id"] = runs_raw["run_id"].astype("string")

print(events_raw.head())
print()
print(events_raw.dtypes)

In [ ]:
config_cuts = {
    "min_energy_GeV": 2.0,
    "max_chi2": 3.0,
}

merged = events_raw.merge(
    runs_raw,
    on="run_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

numeric_cols = ["energy_GeV", "chi2", "signal", "exposure_s"]

finite = np.isfinite(merged[numeric_cols]).all(axis=1)
physical = (
    (merged["energy_GeV"] >= config_cuts["min_energy_GeV"])
    & (merged["chi2"] <= config_cuts["max_chi2"])
    & (merged["exposure_s"] > 0.0)
)

clean_events = (
    merged.loc[finite & physical & (merged["_merge"] == "both")]
    .copy()
    .assign(signal_rate=lambda d: d["signal"] / d["exposure_s"])
)

print("raw rows:", len(events_raw))
print("clean rows:", len(clean_events))
print("removed rows:", len(events_raw) - len(clean_events))
print()
print(clean_events.head())

In [ ]:
detector_summary = (
    clean_events.groupby("detector", observed=True, as_index=False)
    .agg(
        n_events=("event_id", "size"),
        mean_energy_GeV=("energy_GeV", "mean"),
        mean_signal_rate=("signal_rate", "mean"),
        std_signal_rate=("signal_rate", "std"),
    )
)

detector_summary

Questions:

1. Which columns are numerical measurements, and which columns are labels or metadata?
2. Which cleaning rules are scientific assumptions rather than technical details?
3. Why does `validate="many_to_one"` protect the join with run metadata?

### Example 4 - A small reproducible analysis pipeline

The same operations can be organized into a tiny pipeline. The goal is not to build a large framework. The goal is to make the flow from raw files to cleaned data, summary tables, and diagnostic plots explicit.

In [ ]:
project_root = Path("demo_pipeline")
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

events_raw.to_csv(raw_dir / "events.csv", index=False)
runs_raw.to_csv(raw_dir / "runs.csv", index=False)

pipeline_config = {
    "min_energy_GeV": 2.0,
    "max_chi2": 3.0,
}

In [ ]:
def load_inputs(project_root):
    raw_dir = project_root / "data" / "raw"

    events = pd.read_csv(
        raw_dir / "events.csv",
        dtype={
            "run_id": "string",
            "detector": "category",
        },
    )

    runs = pd.read_csv(
        raw_dir / "runs.csv",
        dtype={"run_id": "string"},
    )

    return events, runs


def clean_event_table(events, runs, config):
    merged = events.merge(
        runs,
        on="run_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    numeric_cols = ["energy_GeV", "chi2", "signal", "exposure_s"]

    finite = np.isfinite(merged[numeric_cols]).all(axis=1)
    cuts = (
        (merged["energy_GeV"] >= config["min_energy_GeV"])
        & (merged["chi2"] <= config["max_chi2"])
        & (merged["exposure_s"] > 0.0)
    )
    has_metadata = merged["_merge"] == "both"

    clean = (
        merged.loc[finite & cuts & has_metadata]
        .copy()
        .assign(signal_rate=lambda d: d["signal"] / d["exposure_s"])
    )

    diagnostics = {
        "n_raw": len(events),
        "n_clean": len(clean),
        "n_missing_metadata": int((~has_metadata).sum()),
    }

    return clean, diagnostics


def summarize_by_run_and_detector(clean):
    return (
        clean.groupby(["run_id", "detector"], as_index=False, observed=True)
        .agg(
            n_events=("event_id", "size"),
            mean_energy_GeV=("energy_GeV", "mean"),
            mean_rate=("signal_rate", "mean"),
            std_rate=("signal_rate", "std"),
            field_T=("field_T", "first"),
        )
    )

In [ ]:
def save_outputs(summary, config, diagnostics, project_root):
    table_dir = project_root / "results" / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)

    summary.to_csv(table_dir / "summary.csv", index=False)

    metadata = {
        "config": config,
        "diagnostics": diagnostics,
    }

    (table_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf8",
    )


def plot_summary(summary, project_root):
    fig_dir = project_root / "results" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    by_field = (
        summary.groupby("field_T", as_index=False)
        .agg(mean_rate=("mean_rate", "mean"))
        .sort_values("field_T")
    )

    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.plot(by_field["field_T"], by_field["mean_rate"], "o-")

    ax.set_xlabel(r"magnetic field $B$ [T]")
    ax.set_ylabel(r"mean signal rate")
    ax.grid(True, alpha=0.3)

    fig.savefig(fig_dir / "signal_rate_by_field.pdf", bbox_inches="tight")

    return fig, ax


def run_pipeline(project_root, config):
    events, runs = load_inputs(project_root)
    clean, diagnostics = clean_event_table(events, runs, config)
    summary = summarize_by_run_and_detector(clean)

    save_outputs(summary, config, diagnostics, project_root)
    fig, ax = plot_summary(summary, project_root)

    return clean, summary, diagnostics, fig, ax


clean_pipeline, summary_pipeline, diagnostics_pipeline, fig, ax = run_pipeline(
    project_root,
    pipeline_config,
)

diagnostics_pipeline, summary_pipeline.head()

Questions:

1. Which function is responsible for each stage of the pipeline?
2. Which outputs would allow a collaborator to understand how the summary table was produced?
3. What would you change if the cleaning cuts had to be varied over many parameter values?

## Part II - Independent work

### Task 1 - Make state and mutation explicit

Use the following setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_global = np.random.default_rng(2026)
sigma_noise = 0.08

t = np.linspace(0.0, 8.0, 300)
reference = np.exp(-0.30 * t) * np.cos(4.0 * t)


def noisy_measurement_bad(signal):
    signal += sigma_noise * rng_global.normal(size=signal.shape)
    return signal


raw = reference.copy()
measured_bad = noisy_measurement_bad(raw)

print("raw still equal to reference?", np.allclose(raw, reference))

Your tasks are:

1. Identify the two hidden dependencies or side effects in `noisy_measurement_bad`.
2. Write a safer function `add_noise(signal, *, sigma, rng)` that returns a new array and does not modify `signal`.
3. Use a fixed seed to generate a reproducible noisy measurement.
4. Add checks that the original `reference` array was not changed and that the noisy measurement is finite.
5. Plot the clean reference signal and one noisy measurement on the same axes.

**Optional extension**: Write a second helper function called `add_noise_in_place(signal, *, sigma, rng)` that intentionally modifies its input array. Use it only on `reference.copy()`, not on `reference` itself. Compare the result with the safer `add_noise` function and write one sentence explaining when an in-place version might be justified and why the function name should make mutation explicit.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Basic array contracts
# ------------------------------------------------------------

assert t.ndim == 1, f"t must be one-dimensional, but got t.ndim = {t.ndim}"
assert reference.ndim == 1, (
    f"reference must be one-dimensional, but got reference.ndim = {reference.ndim}"
)

assert t.shape == reference.shape
assert np.isfinite(t).all()
assert np.isfinite(reference).all()

reference_original = reference.copy()


# ------------------------------------------------------------
# Point 2
# Safer noise-adding function
# ------------------------------------------------------------

def add_noise(signal, *, sigma, rng):
    if signal.ndim != 1:
        raise ValueError(
            f"signal must be one-dimensional, but got signal.ndim = {signal.ndim}"
        )

    if sigma < 0.0:
        raise ValueError(
            f"sigma must be non-negative, but got sigma = {sigma}"
        )

    if rng is None:
        raise ValueError("rng must be provided explicitly")

    if not np.isfinite(signal).all():
        raise ValueError("signal must contain only finite values")

    noise = sigma * rng.normal(size=signal.shape)

    # This creates and returns a new array.
    # The input signal is not modified.
    return signal + noise


# ------------------------------------------------------------
# Point 3
# Reproducible noisy measurement with a fixed seed
# ------------------------------------------------------------

rng = np.random.default_rng(123)

measured = add_noise(
    reference,
    sigma=sigma_noise,
    rng=rng,
)


# ------------------------------------------------------------
# Point 4
# Checks: reference unchanged and measured signal finite
# ------------------------------------------------------------

assert np.allclose(reference, reference_original)
assert np.isfinite(measured).all()
assert measured.shape == reference.shape

print("reference unchanged:", np.allclose(reference, reference_original))
print("measured finite:    ", np.isfinite(measured).all())
print("measured shape:     ", measured.shape)


# ------------------------------------------------------------
# Point 5
# Plot clean reference and noisy measurement
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 3.8))

ax.plot(
    t,
    reference,
    linewidth=2,
    label="clean reference",
)

ax.plot(
    t,
    measured,
    ".",
    markersize=3,
    label="noisy measurement",
)

ax.set_xlabel(r"time $t$")
ax.set_ylabel("signal")
ax.set_title("Reference signal and reproducible noisy measurement")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
#####################
# Optional extension
#####################

# ------------------------------------------------------------
# Intentionally in-place version
# ------------------------------------------------------------

def add_noise_in_place(signal, *, sigma, rng):
    if signal.ndim != 1:
        raise ValueError(
            f"signal must be one-dimensional, but got signal.ndim = {signal.ndim}"
        )

    if sigma < 0.0:
        raise ValueError(
            f"sigma must be non-negative, but got sigma = {sigma}"
        )

    if rng is None:
        raise ValueError("rng must be provided explicitly")

    if not np.isfinite(signal).all():
        raise ValueError("signal must contain only finite values")

    signal += sigma * rng.normal(size=signal.shape)

    # Return the same modified object for convenience.
    return signal


# Use the in-place function only on a copy.
reference_for_in_place = reference.copy()

rng_safe = np.random.default_rng(123)
rng_in_place = np.random.default_rng(123)

measured_safe = add_noise(
    reference,
    sigma=sigma_noise,
    rng=rng_safe,
)

measured_in_place = add_noise_in_place(
    reference_for_in_place,
    sigma=sigma_noise,
    rng=rng_in_place,
)

assert np.allclose(reference, reference_original)
assert np.allclose(measured_safe, measured_in_place)
assert measured_in_place is reference_for_in_place

print("reference unchanged after in-place test:",
      np.allclose(reference, reference_original))
print("safe and in-place results equal:        ",
      np.allclose(measured_safe, measured_in_place))
print("in-place result is the modified input:  ",
      measured_in_place is reference_for_in_place)


fig, ax = plt.subplots(figsize=(7.0, 3.8))

ax.plot(
    t,
    reference,
    linewidth=2,
    label="clean reference",
)

ax.plot(
    t,
    measured_safe,
    ".",
    markersize=3,
    label="safe add_noise",
)

ax.plot(
    t,
    measured_in_place,
    "--",
    linewidth=1,
    label="add_noise_in_place on copy",
)

ax.set_xlabel(r"time $t$")
ax.set_ylabel("signal")
ax.set_title("Safe and intentionally in-place noise addition")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

### Task 2 - Clean and summarize a labelled measurement table

This is a short task. The goal is to treat cleaning rules as visible scientific choices, not as hidden edits to a table.

Use the following setup:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(1234)

n = 240

df = pd.DataFrame({
    "event_id": np.arange(n),
    "run_id": rng.choice(["run_01", "run_02", "run_03"], size=n),
    "detector": rng.choice(["A", "B", "C"], size=n),
    "energy_GeV": rng.gamma(shape=3.5, scale=1.0, size=n),
    "chi2": rng.gamma(shape=1.5, scale=0.7, size=n),
    "signal": rng.normal(loc=10.0, scale=1.5, size=n),
    "exposure_s": rng.uniform(0.4, 1.8, size=n),
})

df.loc[4, "signal"] = np.nan
df.loc[15, "energy_GeV"] = -1.0
df.loc[31, "exposure_s"] = 0.0
df.loc[58, "chi2"] = np.inf

df["run_id"] = df["run_id"].astype("string")
df["detector"] = df["detector"].astype("category")

config = {
    "min_energy_GeV": 1.5,
    "max_chi2": 2.5,
}

Your tasks are:

1. Inspect `df.head()`, `df.dtypes`, and the number of missing values per column.
2. Create a cleaned table called `clean` by requiring finite numerical values, positive exposure, `energy_GeV >= config["min_energy_GeV"]`, and `chi2 <= config["max_chi2"]`.
3. Add a derived column `signal_rate = signal / exposure_s`.
4. Compute a summary by detector with the number of events, mean signal rate, and standard deviation of the signal rate.
5. Report how many rows were removed, and make one labelled diagnostic plot from the summary.

**Optional extension**: Repeat the cleaning and summary for two different values of `max_chi2`, for example `2.0` and `3.0`. Compare how many rows are retained and how the mean signal rate changes for each detector. Does the scientific conclusion look stable under this reasonable change of the cut?

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Inspect the table
# ------------------------------------------------------------

display(df.head())

print("Data types:")
display(df.dtypes)

print("Missing values per column:")
display(df.isna().sum())


# ------------------------------------------------------------
# Point 2
# Create a cleaned table using visible scientific cuts
# ------------------------------------------------------------

numeric_cols = [
    "energy_GeV",
    "chi2",
    "signal",
    "exposure_s",
]

finite_numeric = np.isfinite(df[numeric_cols]).all(axis=1)

positive_exposure = df["exposure_s"] > 0.0

energy_cut = df["energy_GeV"] >= config["min_energy_GeV"]

chi2_cut = df["chi2"] <= config["max_chi2"]

clean_mask = (
    finite_numeric
    & positive_exposure
    & energy_cut
    & chi2_cut
)

clean = df.loc[clean_mask].copy()

assert clean.shape[0] <= df.shape[0]
assert np.isfinite(clean[numeric_cols]).all(axis=None)
assert (clean["exposure_s"] > 0.0).all()
assert (clean["energy_GeV"] >= config["min_energy_GeV"]).all()
assert (clean["chi2"] <= config["max_chi2"]).all()


# ------------------------------------------------------------
# Point 3
# Add derived column
# ------------------------------------------------------------

clean["signal_rate"] = clean["signal"] / clean["exposure_s"]

assert "signal_rate" in clean.columns
assert np.isfinite(clean["signal_rate"]).all()


# ------------------------------------------------------------
# Point 4
# Summary by detector
# ------------------------------------------------------------

summary = (
    clean
    .groupby("detector", as_index=False, observed=True)
    .agg(
        n_events=("event_id", "size"),
        mean_signal_rate=("signal_rate", "mean"),
        std_signal_rate=("signal_rate", "std"),
    )
)

display(summary)


# ------------------------------------------------------------
# Point 5
# Report removed rows and make a labelled diagnostic plot
# ------------------------------------------------------------

n_removed = df.shape[0] - clean.shape[0]

print("Rows before cleaning:", df.shape[0])
print("Rows after cleaning: ", clean.shape[0])
print("Rows removed:        ", n_removed)


fig, ax = plt.subplots(figsize=(5.6, 3.8))

ax.errorbar(
    summary["detector"].astype(str),
    summary["mean_signal_rate"],
    yerr=summary["std_signal_rate"],
    fmt="o",
    capsize=4,
)

ax.set_xlabel("detector")
ax.set_ylabel("signal rate [signal / s]")
ax.set_title("Mean signal rate by detector after cleaning")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
#####################
# Optional extension
#####################

max_chi2_values = [2.0, 3.0]

comparison_tables = []

for max_chi2 in max_chi2_values:
    finite_numeric = np.isfinite(df[numeric_cols]).all(axis=1)

    clean_mask = (
        finite_numeric
        & (df["exposure_s"] > 0.0)
        & (df["energy_GeV"] >= config["min_energy_GeV"])
        & (df["chi2"] <= max_chi2)
    )

    clean_cut = df.loc[clean_mask].copy()
    clean_cut["signal_rate"] = clean_cut["signal"] / clean_cut["exposure_s"]

    summary_cut = (
        clean_cut
        .groupby("detector", as_index=False, observed=True)
        .agg(
            n_events=("event_id", "size"),
            mean_signal_rate=("signal_rate", "mean"),
            std_signal_rate=("signal_rate", "std"),
        )
    )

    summary_cut["max_chi2"] = max_chi2
    summary_cut["n_retained_total"] = clean_cut.shape[0]

    comparison_tables.append(summary_cut)

comparison = pd.concat(
    comparison_tables,
    ignore_index=True,
)

display(comparison)


fig, ax = plt.subplots(figsize=(6.2, 3.8))

for max_chi2, table in comparison.groupby("max_chi2"):
    ax.plot(
        table["detector"].astype(str),
        table["mean_signal_rate"],
        "o-",
        label=fr"max chi2 = {max_chi2}",
    )

ax.set_xlabel("detector")
ax.set_ylabel("mean signal rate [signal / s]")
ax.set_title("Sensitivity of detector summary to chi2 cut")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

### Task 3 - A small reproducible event-analysis pipeline

This is the longer task, but it should still stay compact. The goal is to connect explicit functions, portable paths, labelled tables, validated joins, and saved outputs.

The setup below creates two raw CSV files:

```text
events.csv       event-level measurements
runs.csv         run-level metadata
```

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(4321)

project_root = Path("student_pipeline")
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

run_ids = [f"run_{i:03d}" for i in range(1, 7)]
detectors = ["A", "B", "C"]

runs = pd.DataFrame({
    "run_id": run_ids,
    "field_T": np.linspace(0.05, 0.35, len(run_ids)),
    "temperature_K": 300.0 + rng.normal(scale=0.4, size=len(run_ids)),
})

n_events = 900

events = pd.DataFrame({
    "event_id": np.arange(n_events),
    "run_id": rng.choice(run_ids, size=n_events),
    "detector": rng.choice(detectors, size=n_events),
    "energy_GeV": rng.gamma(shape=4.0, scale=0.9, size=n_events),
    "chi2": rng.gamma(shape=1.4, scale=0.8, size=n_events),
    "signal": rng.normal(loc=15.0, scale=2.5, size=n_events),
    "exposure_s": rng.uniform(0.5, 2.0, size=n_events),
})

# A few deliberately problematic rows.
events.loc[2, "run_id"] = "run_missing"
events.loc[7, "signal"] = np.nan
events.loc[19, "energy_GeV"] = -2.0
events.loc[41, "chi2"] = np.inf
events.loc[62, "exposure_s"] = 0.0

events.to_csv(raw_dir / "events.csv", index=False)
runs.to_csv(raw_dir / "runs.csv", index=False)

project_root

Your tasks are:

1. Define a `config` dictionary containing at least `min_energy_GeV` and `max_chi2`.
2. Write `load_inputs(project_root)` that reads both CSV files using `pathlib` paths and sets useful dtypes for `run_id` and `detector`.
3. Write `clean_events(events, runs, config)` that:
   - joins events with run metadata using `validate="many_to_one"`;
   - removes rows with missing metadata;
   - removes non-finite or physically invalid numerical rows;
   - applies the energy and chi-square cuts;
   - adds `signal_rate`.
4. Write `summarize(clean)` that groups by `run_id` and `detector`, and returns at least `n_events`, `mean_energy_GeV`, `mean_rate`, and `field_T`.
5. Save the summary table and the configuration to `results/tables`.
6. Make one diagnostic plot of mean signal rate versus magnetic field.
7. Write a short interpretation: which assumptions are visible in the pipeline, and what rows were removed?

**Optional extension**: Add one small audit table to the pipeline. It should record how many rows were removed because of missing metadata, non-finite or invalid numerical values, and the final energy/chi-square cuts. Save this table together with the summary and configuration. In one or two sentences, explain why this audit table is useful for reproducibility.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Visible analysis configuration
# ------------------------------------------------------------

config = {
    "min_energy_GeV": 1.5,
    "max_chi2": 2.5,
}


# ------------------------------------------------------------
# Point 2
# Load raw inputs using pathlib paths and useful dtypes
# ------------------------------------------------------------

def load_inputs(project_root):
    raw_dir = project_root / "data" / "raw"

    events_path = raw_dir / "events.csv"
    runs_path = raw_dir / "runs.csv"

    events = pd.read_csv(
        events_path,
        dtype={
            "run_id": "string",
            "detector": "category",
        },
    )

    runs = pd.read_csv(
        runs_path,
        dtype={
            "run_id": "string",
        },
    )

    return events, runs


# ------------------------------------------------------------
# Point 3
# Clean event table and optionally return an audit table
# ------------------------------------------------------------

def clean_events(events, runs, config, *, return_audit=False):
    merged = events.merge(
        runs,
        on="run_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    n_input = merged.shape[0]

    # Remove events whose run_id was not found in runs.csv.
    has_metadata = merged["_merge"] == "both"
    with_metadata = merged.loc[has_metadata].copy()

    n_missing_metadata = n_input - with_metadata.shape[0]

    numerical_cols = [
        "energy_GeV",
        "chi2",
        "signal",
        "exposure_s",
        "field_T",
        "temperature_K",
    ]

    finite_numerical = np.isfinite(with_metadata[numerical_cols]).all(axis=1)

    physically_valid = (
        (with_metadata["energy_GeV"] >= 0.0)
        & (with_metadata["chi2"] >= 0.0)
        & (with_metadata["exposure_s"] > 0.0)
    )

    valid_numerical = finite_numerical & physically_valid
    valid = with_metadata.loc[valid_numerical].copy()

    n_invalid_numerical = with_metadata.shape[0] - valid.shape[0]

    analysis_cuts = (
        (valid["energy_GeV"] >= config["min_energy_GeV"])
        & (valid["chi2"] <= config["max_chi2"])
    )

    clean = valid.loc[analysis_cuts].copy()

    n_removed_by_cuts = valid.shape[0] - clean.shape[0]

    clean["signal_rate"] = clean["signal"] / clean["exposure_s"]

    clean = clean.drop(columns=["_merge"])

    # Basic contracts for the cleaned table.
    assert clean["run_id"].notna().all()
    assert np.isfinite(clean[numerical_cols + ["signal_rate"]]).all(axis=None)
    assert (clean["exposure_s"] > 0.0).all()
    assert (clean["energy_GeV"] >= config["min_energy_GeV"]).all()
    assert (clean["chi2"] <= config["max_chi2"]).all()

    if return_audit:
        audit = pd.DataFrame({
            "stage": [
                "input rows",
                "removed: missing metadata",
                "removed: non-finite or invalid numerical values",
                "removed: energy or chi2 cuts",
                "final clean rows",
            ],
            "n_rows": [
                n_input,
                n_missing_metadata,
                n_invalid_numerical,
                n_removed_by_cuts,
                clean.shape[0],
            ],
        })

        return clean, audit

    return clean


# ------------------------------------------------------------
# Point 4
# Summarize by run_id and detector
# ------------------------------------------------------------

def summarize(clean):
    summary = (
        clean
        .groupby(["run_id", "detector"], as_index=False, observed=True)
        .agg(
            n_events=("event_id", "size"),
            mean_energy_GeV=("energy_GeV", "mean"),
            mean_rate=("signal_rate", "mean"),
            std_rate=("signal_rate", "std"),
            field_T=("field_T", "first"),
            temperature_K=("temperature_K", "first"),
        )
        .sort_values(["run_id", "detector"])
        .reset_index(drop=True)
    )

    return summary


# ------------------------------------------------------------
# Point 5
# Save summary table, configuration, and optional audit table
# ------------------------------------------------------------

def save_outputs(summary, config, project_root, *, audit=None):
    table_dir = project_root / "results" / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)

    summary_path = table_dir / "run_detector_summary.csv"
    config_path = table_dir / "config.json"

    summary.to_csv(summary_path, index=False)

    config_path.write_text(
        json.dumps(config, indent=2),
        encoding="utf8",
    )

    paths = {
        "summary": summary_path,
        "config": config_path,
    }

    if audit is not None:
        audit_path = table_dir / "cleaning_audit.csv"
        audit.to_csv(audit_path, index=False)
        paths["audit"] = audit_path

    return paths


# ------------------------------------------------------------
# Point 6
# Diagnostic plot of mean signal rate versus magnetic field
# ------------------------------------------------------------

def plot_mean_rate(summary, project_root):
    fig, ax = plt.subplots(figsize=(6.4, 4.0))

    for detector, table in summary.groupby("detector", observed=True):
        table = table.sort_values("field_T")

        ax.plot(
            table["field_T"],
            table["mean_rate"],
            "o-",
            label=f"detector {detector}",
        )

    ax.set_xlabel("magnetic field [T]")
    ax.set_ylabel("mean signal rate [signal / s]")
    ax.set_title("Mean signal rate versus magnetic field")
    ax.grid(True, alpha=0.3)
    ax.legend(title="detector")

    fig.tight_layout()

    figure_dir = project_root / "results" / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    figure_path = figure_dir / "mean_rate_vs_field.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")

    return fig, ax, figure_path

In [ ]:
# ------------------------------------------------------------
# Run the pipeline
# ------------------------------------------------------------

events_loaded, runs_loaded = load_inputs(project_root)

clean, audit = clean_events(
    events_loaded,
    runs_loaded,
    config,
    return_audit=True,
)

summary = summarize(clean)

output_paths = save_outputs(
    summary,
    config,
    project_root,
    audit=audit,
)

fig, ax, figure_path = plot_mean_rate(
    summary,
    project_root,
)

display(audit)
display(summary.head())

print("Input rows:       ", events_loaded.shape[0])
print("Clean rows:       ", clean.shape[0])
print("Rows removed:     ", events_loaded.shape[0] - clean.shape[0])
print()
print("Saved files:")
for name, path in output_paths.items():
    print(f"{name:>8s}: {path}")
print(f"{'figure':>8s}: {figure_path}")

plt.show()